# Cambridge and MassDOT: November 2025

Load the November 2025 MassDOT records and the Cambridge Police November 2025 records into separate pandas dataframes for later comparison. This notebook does not merge the dataframes or write any output files.

The data folder currently contains `boston-city-from-massdot.csv` rather than `boston-from-massdot.csv`, so the available `boston-city-from-massdot.csv` file is used below.

In [8]:
from pathlib import Path

import pandas as pd

MASSDOT_FILE = Path("boston-city-from-massdot.csv")
CAMBRIDGE_FILE = Path("cambridge-police-nov-25.csv")

In [9]:
massdot_df = pd.read_csv(MASSDOT_FILE, parse_dates=["date"])
massdot_nov_2025 = massdot_df.loc[
    (massdot_df["date"] >= "2025-11-01")
    & (massdot_df["date"] < "2025-12-01")
].copy()

cambridge_nov_2025 = pd.read_csv(CAMBRIDGE_FILE, parse_dates=["date"])

print(f"MassDOT November 2025 rows: {len(massdot_nov_2025):,}")
print(f"Cambridge November 2025 rows: {len(cambridge_nov_2025):,}")

MassDOT November 2025 rows: 364
Cambridge November 2025 rows: 118


## Preview the separate dataframes

In [10]:
display(massdot_nov_2025.head())
display(cambridge_nov_2025.head())

,Unnamed: 0,source,id,muni,year,date,severity,time,police,pedestrian,cyclist,other,lat,lng,interstate,NON_MTRST_TYPE_CL,MOST_HRMFL_EVT_CL
24219,24219,MassDOT,5591245,BOSTON,2025,2025-11-01,2,06:33:00,Local,0,0,0,42.280687,-71.158154,0,NaN,V1:(Collision with motor vehicle in traffic) /...
24220,24220,MassDOT,5591241,BOSTON,2025,2025-11-01,0,08:44:00,Local,0,0,0,42.337310,-71.152445,0,NaN,V1:(Collision with parked motor vehicle)
24234,24234,MassDOT,5591355,BOSTON,2025,2025-11-01,0,20:00:00,Local,0,0,0,42.316998,-71.100244,0,NaN,V1:(Collision with parked motor vehicle)
24236,24236,MassDOT,5591331,BOSTON,2025,2025-11-02,0,02:00:00,Local,0,0,0,42.293065,-71.094490,0,NaN,V1:(Collision with motor vehicle in traffic) /...
24237,24237,MassDOT,5591328,BOSTON,2025,2025-11-01,2,23:20:00,Local,0,0,0,42.314778,-71.097224,0,NaN,NaN


,source,id,muni,year,date,severity,time,police,cyclist,pedestrian,other,lat,lng,street_name,datetime
0,CambridgePD,CPD_15738,CAMBRIDGE,2025,2025-11-01,0,00:46:00,NaN,0,0,0,NaN,NaN,THIRD STREET,2025-11-01 00:46:00
1,CambridgePD,CPD_15739,CAMBRIDGE,2025,2025-11-01,0,16:05:00,NaN,1,0,0,42.37162,-71.09761,HAMPSHIRE ST,2025-11-01 16:05:00
2,CambridgePD,CPD_15740,CAMBRIDGE,2025,2025-11-01,2,17:00:00,NaN,0,0,0,42.37363,-71.10007,CAMBRDGE STREET,2025-11-01 17:00:00
3,CambridgePD,CPD_15741,CAMBRIDGE,2025,2025-11-01,0,17:00:00,NaN,0,0,0,42.36473,-71.10408,GREEN STREET,2025-11-01 17:00:00
4,CambridgePD,CPD_15742,CAMBRIDGE,2025,2025-11-01,0,17:32:00,NaN,0,0,0,42.38862,-71.12362,RICHDALE AVE,2025-11-01 17:32:00


## Combine the November 2025 records

Align the shared fields by column name, discard the two MassDOT-only classification columns, and include the requested columns after `lng` in this order: `interstate`, `street_name`, `datetime`. The source dataframes remain separate; the combined dataframe is saved as `cambridge-boston-merged.csv`.

In [12]:
base_columns = [
    "source", "id", "muni", "year", "date", "severity", "time", "police",
    "pedestrian", "cyclist", "other", "lat", "lng",
]
trailing_columns = ["interstate", "street_name", "datetime"]
output_columns = base_columns + trailing_columns

massdot_for_merge = massdot_nov_2025.drop(
    columns=["Unnamed: 0", "NON_MTRST_TYPE_CL", "MOST_HRMFL_EVT_CL"],
    errors="ignore",
).copy()
cambridge_for_merge = cambridge_nov_2025.copy()

for dataframe_name, dataframe in [("MassDOT", massdot_for_merge), ("Cambridge", cambridge_for_merge)]:
    missing_columns = [column for column in base_columns if column not in dataframe.columns]
    if missing_columns:
        raise ValueError(f"{dataframe_name} dataframe is missing shared columns: {missing_columns}")

for column in trailing_columns:
    if column not in massdot_for_merge.columns:
        massdot_for_merge[column] = pd.NA
    if column not in cambridge_for_merge.columns:
        cambridge_for_merge[column] = pd.NA

merged_nov_2025 = pd.concat(
    [
        massdot_for_merge.reindex(columns=output_columns),
        cambridge_for_merge.reindex(columns=output_columns),
    ],
    ignore_index=True,
)

OUTPUT_FILE = Path("cambridge-boston-merged.csv")
merged_nov_2025.to_csv(OUTPUT_FILE, index=False)

print(f"Merged rows: {len(merged_nov_2025):,}")
print(f"Merged columns: {merged_nov_2025.columns.tolist()}")
print(f"Saved merged data to: {OUTPUT_FILE.resolve()}")
display(merged_nov_2025.head())

PermissionError: [Errno 13] Permission denied: 'cambridge-boston-merged.csv'

## Check Boston rows for duplicate crash records

Within the `muni == "BOSTON"` rows, mark each repeated record after its first occurrence as a duplicate. A match requires exact equality across `date`, `severity`, and `time`.

In [13]:
duplicate_key_columns = ["date", "severity", "time"]
boston_rows = merged_nov_2025.loc[merged_nov_2025["muni"].eq("BOSTON")].copy()
duplicate_massdot_mask = boston_rows.duplicated(
    subset=duplicate_key_columns,
    keep="first",
)
duplicate_massdot_rows = boston_rows.loc[duplicate_massdot_mask].copy()
duplicate_massdot_count = len(duplicate_massdot_rows)

print(f"Duplicate MassDOT rows (additional matching rows): {duplicate_massdot_count}")
display(duplicate_massdot_rows)

Duplicate MassDOT rows (additional matching rows): 0


,source,id,muni,year,date,severity,time,police,pedestrian,cyclist,other,lat,lng,interstate,street_name,datetime
